In [1]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score
import torchvision.transforms as transforms
from PIL import Image

# =====================================================================
# GLOBAL SEED IMPLEMENTATION & REPRODUCIBILITY ENVIRONMENT
# =====================================================================
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =====================================================================
# DATASET STAGE: SCAFFOLD SCANNERS & SUBSAMPLING LOGIC
# =====================================================================
def get_image_files(directory):
    """Helper to list all images in a directory efficiently."""
    directory = directory.strip()
    if not os.path.exists(directory):
        print(f"⚠️ Warning: Directory does not exist -> '{directory}'")
        return []
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, ext)))
        files.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(files)

# Master storage registry for multi-dataset balancing
metadata_records = []

print("=== STARTING DATASET SCAN AND SUBSAMPLING ===")

# --- 1. DF2023 Train Subset (Strict Limit: 10,000 items) ---
df2023_train_img_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15"
df2023_train_gt_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15_GT"

df2023_train_imgs = get_image_files(df2023_train_img_dir)
print(f"Found {len(df2023_train_imgs)} files in DF2023 Train")

if df2023_train_imgs:
    train_sample_size = min(10000, len(df2023_train_imgs))
    sampled_train_imgs = np.random.choice(df2023_train_imgs, size=train_sample_size, replace=False)
    
    for img_path in sampled_train_imgs:
        base_name = os.path.basename(img_path)
        gt_path = os.path.join(df2023_train_gt_dir, base_name)
        metadata_records.append({
            'image_path': img_path,
            'mask_path': gt_path if os.path.exists(gt_path) else None,
            'source_dataset': 'DF2023_Train',
            'category': 'manipulation',
            'split': 'train'
        })

# --- 2. DF2023 Validation & Internal Test Subset (Strict Limit: 3,000 items) ---
df2023_val_img_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15"
df2023_val_gt_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15_GT"

df2023_val_imgs = get_image_files(df2023_val_img_dir)
print(f"Found {len(df2023_val_imgs)} files in DF2023 Val")

if df2023_val_imgs:
    val_sample_size = min(3000, len(df2023_val_imgs))
    sampled_val_imgs = np.random.choice(df2023_val_imgs, size=val_sample_size, replace=False)
    
    df2023_v_idx, df2023_t_idx = train_test_split(sampled_val_imgs, test_size=0.50, random_state=42)
    
    for img_list, split_label in [(df2023_v_idx, 'val'), (df2023_t_idx, 'test')]:
        for img_path in img_list:
            base_name = os.path.basename(img_path)
            gt_path = os.path.join(df2023_val_gt_dir, base_name) 
            metadata_records.append({
                'image_path': img_path,
                'mask_path': gt_path if os.path.exists(gt_path) else None,
                'source_dataset': 'DF2023_Val',
                'category': 'manipulation',
                'split': split_label
            })

# --- 3. SIDA Forensics Subset (Limit to 6,000 items total -> 70/15/15 Split) ---
sida_paths = {
    'full_synthetic': "/kaggle/input/datasets/mubarekahmed/sida-forensics/full_synthetic",
    'masks': "/kaggle/input/datasets/mubarekahmed/sida-forensics/masks",
    'real': "/kaggle/input/datasets/mubarekahmed/sida-forensics/real",
    'tampered': "/kaggle/input/datasets/mubarekahmed/sida-forensics/tampered"
}

sida_all_records = []
for category, path in sida_paths.items():
    sida_imgs = get_image_files(path)
    print(f"Found {len(sida_imgs)} files in SIDA category: {category}")
    for img_path in sida_imgs:
        sida_all_records.append({
            'image_path': img_path,
            'mask_path': None,
            'source_dataset': 'sida-forensics',
            'category': category
        })

sida_df_pool = pd.DataFrame(sida_all_records)
if not sida_df_pool.empty:
    _, sampled_sida_df = train_test_split(
        sida_df_pool, 
        test_size=min(6000, len(sida_df_pool)), 
        stratify=sida_df_pool['category'], 
        random_state=42
    )
    
    sida_train, sida_temp = train_test_split(sampled_sida_df, test_size=0.30, stratify=sampled_sida_df['category'], random_state=42)
    sida_val, sida_test = train_test_split(sida_temp, test_size=0.50, stratify=sida_temp['category'], random_state=42)
    
    sida_train['split'] = 'train'
    sida_val['split'] = 'val'
    sida_test['split'] = 'test'
    
    metadata_records.extend(pd.concat([sida_train, sida_val, sida_test]).to_dict(orient='records'))

# --- 4. So-Fake Setted Combined Subset (Read Parquet Files -> 2k items per split) ---
so_fake_paths = {
    'train': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/train",
    'validation': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/validation",
    'ood_test': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/ood_test"
}

for split_name, path in so_fake_paths.items():
    path = path.strip()
    parquet_files = sorted(glob.glob(os.path.join(path, "*.parquet")))
    print(f"Found {len(parquet_files)} parquet shards in So-Fake: {split_name}")
    
    if parquet_files:
        split_records = []
        for p_file in parquet_files:
            if len(split_records) >= 2000:
                break
            
            df_parquet = pd.read_parquet(p_file)
            
            for _, row in df_parquet.iterrows():
                if len(split_records) >= 2000:
                    break
                
                img_p = row.get('image_path', row.get('path', None))
                mask_p = row.get('mask_path', row.get('mask', None))
                cat = row.get('category', 'synthetic_hybrid')
                
                split_records.append({
                    'image_path': img_p if img_p else p_file,
                    'mask_path': mask_p if pd.notna(mask_p) else None,
                    'source_dataset': 'so-fake-combined',
                    'category': cat,
                    'split': 'val' if split_name == 'validation' else ('test' if split_name == 'ood_test' else 'train')
                })
        
        metadata_records.extend(split_records)

# Safe architecture validation structure matrix conversion
columns = ['image_path', 'mask_path', 'source_dataset', 'category', 'split']
final_metadata_df = pd.DataFrame(metadata_records, columns=columns)
final_metadata_df.to_csv("balanced_dataset_metadata.csv", index=False)

print("\n=== FINAL GENERATED METADATA MATRIX ===")
if not final_metadata_df.empty:
    print(final_metadata_df.groupby(['source_dataset', 'split']).size())
else:
    print("⚠️ Warning: Dataset processing ended with 0 total items during current directory sweep.")

# =====================================================================
# MODEL STAGE: CORE FORENSIC ENGINE AND COMPONENT SUBSYSTEMS
# =====================================================================
def get_srm_filters():
    """Generates three standard Spatial Rich Model (SRM) high-pass kernels."""
    filter1 = np.array([[0,  0, 0], [0, -1, 1], [0,  0, 0]], dtype=np.float32)
    filter2 = np.array([[0,  1, 0], [0, -2, 1], [0,  0, 0]], dtype=np.float32)
    filter3 = np.array([[0, -1, 0], [-1, 4, -1], [0, -1, 0]], dtype=np.float32)
    
    filters = np.zeros((3, 3, 3, 3), dtype=np.float32)
    for i, f in enumerate([filter1, filter2, filter3]):
        for j in range(3):
            filters[i, j, :, :] = f
    return torch.tensor(filters)

class SRMConvStream(nn.Module):
    """Artifact stream extracting high-pass localized residual microstructures."""
    def __init__(self):
        super(SRMConvStream, self).__init__()
        self.srm_conv = nn.Conv2d(3, 3, kernel_size=3, padding=1, bias=False)
        self.srm_conv.weight.data.copy_(get_srm_filters())
        self.srm_conv.weight.requires_grad = False 
        
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),  # 112x112
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # 56x56
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1), # 28x28
            nn.BatchNorm2d(128),
            nn.SiLU()
        )

    def forward(self, x):
        residuals = self.srm_conv(x)
        features = self.backbone(residuals)
        return features

class TinyViT5MStream(nn.Module):
    """Global Semantic stream collecting macro-level structural anomalies."""
    def __init__(self, embed_dim=128):
        super(TinyViT5MStream, self).__init__()
        self.patch_embed = nn.Conv2d(3, embed_dim, kernel_size=8, stride=8) # 224x224 -> 28x28
        self.norm = nn.LayerNorm(embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=4, dim_feedforward=512, 
            activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

    def forward(self, x):
        patches = self.patch_embed(x) 
        B, C, H, W = patches.shape
        patches = patches.flatten(2).transpose(1, 2) 
        patches = self.norm(patches)
        
        transformed = self.transformer(patches) 
        features = transformed.transpose(1, 2).view(B, C, H, W) 
        return features

class ModalityAwareFusion(nn.Module):
    """Robust multi-modal fusion handling missing metadata via an explicit token."""
    def __init__(self, feature_dim=128, metadata_dim=16):
        super(ModalityAwareFusion, self).__init__()
        self.feature_dim = feature_dim
        
        self.meta_encoder = nn.Sequential(
            nn.Linear(metadata_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, feature_dim)
        )
        self.missing_token = nn.Parameter(torch.randn(1, 1, feature_dim))
        self.cross_attn = nn.MultiheadAttention(embed_dim=feature_dim, num_heads=4, batch_first=True)
        self.norm_visual = nn.LayerNorm(feature_dim)
        self.norm_meta = nn.LayerNorm(feature_dim)

    def forward(self, visual_feats, metadata, meta_present_flags):
        B, C, H, W = visual_feats.shape
        v_flat = visual_feats.flatten(2).transpose(1, 2) 
        
        meta_encoded = self.meta_encoder(metadata).unsqueeze(1) 
        missing_expanded = self.missing_token.expand(B, -1, -1)
        flags = meta_present_flags.view(B, 1, 1).float()
        meta_tokens = (flags * meta_encoded) + ((1.0 - flags) * missing_expanded)
        
        q = self.norm_visual(v_flat)
        k = self.norm_meta(meta_tokens)
        v = meta_tokens
        
        attn_out, _ = self.cross_attn(q, k, v)
        fused_flat = v_flat + attn_out 
        
        fused_feats = fused_flat.transpose(1, 2).view(B, C, H, W)
        return fused_feats

class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super(DepthwiseSeparableConv, self).__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, kernel_size=3, padding=1, stride=stride, groups=in_ch, bias=False)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(self.bn(self.pointwise(self.depthwise(x))))

class FPN_PANet_Head(nn.Module):
    """High-efficiency dual-pyramid layout maximizing pixel localization."""
    def __init__(self, feature_dim=128):
        super(FPN_PANet_Head, self).__init__()
        self.fpn_conv1 = DepthwiseSeparableConv(feature_dim, 64)
        self.fpn_conv2 = DepthwiseSeparableConv(64, 32)
        
        self.pan_conv1 = DepthwiseSeparableConv(32, 64)
        self.pan_conv2 = DepthwiseSeparableConv(64, 128)
        
        self.final_mask = nn.Sequential(
            nn.ConvTranspose2d(128, 32, kernel_size=4, stride=4, padding=0), 
            nn.SiLU(),
            nn.ConvTranspose2d(32, 1, kernel_size=2, stride=2, padding=0),  
            nn.Sigmoid()
        )

    def forward(self, x):
        p1 = self.fpn_conv1(x)                        
        p2 = self.fpn_conv2(F.interpolate(p1, scale_factor=2, mode='nearest')) 
        
        n1 = self.pan_conv1(F.interpolate(p2, scale_factor=0.5, mode='nearest')) + p1
        n2 = self.pan_conv2(n1)
        
        mask = self.final_mask(n2)
        return mask

class FakeShieldT(nn.Module):
    """Complete multi-task dual-stream pipeline architecture."""
    def __init__(self, metadata_dim=16):
        super(FakeShieldT, self).__init__()
        self.artifact_stream = SRMConvStream()
        self.semantic_stream = TinyViT5MStream()
        
        self.fusion = ModalityAwareFusion(feature_dim=128, metadata_dim=metadata_dim)
        self.localization_head = FPN_PANet_Head(feature_dim=128)
        
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.detection_head = nn.Sequential(
            nn.Linear(128, 32),
            nn.SiLU(),
            nn.Linear(32, 2) 
        )

    def forward(self, images, metadata, meta_flags):
        feat_art = self.artifact_stream(images)
        feat_sem = self.semantic_stream(images)
        
        combined_feats = feat_art + feat_sem
        fused_context = self.fusion(combined_feats, metadata, meta_flags)
        
        pooled = self.global_pool(fused_context).view(fused_context.size(0), -1)
        logits_det = self.detection_head(pooled)
        masks_loc = self.localization_head(fused_context)
        
        return logits_det, masks_loc

# =====================================================================
# EVALUATION STAGE: MATHEMATICAL METRICS VERIFICATION ENGINE
# =====================================================================
class FakeShieldEvaluationEngine:
    @staticmethod
    def calculate_miou(pred_masks, gt_masks, threshold=0.5):
        preds = (pred_masks > threshold).float()
        intersection = (preds * gt_masks).sum(dim=(2, 3))
        union = (preds + gt_masks).clamp(0, 1).sum(dim=(2, 3))
        iou = (intersection + 1e-7) / (union + 1e-7)
        return iou.mean().item()

    @staticmethod
    def evaluation_report(all_logits, all_labels, all_pred_masks, all_gt_masks):
        preds_det = torch.argmax(all_logits, dim=1).cpu().numpy()
        labels_det = all_labels.cpu().numpy()
        
        f1 = f1_score(labels_det, preds_det, average='binary', zero_division=0)
        miou = FakeShieldEvaluationEngine.calculate_miou(all_pred_masks, all_gt_masks)
        
        return {"F1_Score": f1, "mIoU": miou}

    @staticmethod
    def compute_robustness_metrics(clean_results, platform_results_dict):
        f1_clean = clean_results["F1_Score"]
        miou_clean = clean_results["mIoU"]
        
        total_delta = 0.0
        num_platforms = len(platform_results_dict)
        report = {}
        
        for platform, res in platform_results_dict.items():
            delta_f1 = f1_clean - res["F1_Score"]
            delta_miou = miou_clean - res["mIoU"]
            report[f'Delta_F1_{platform}'] = delta_f1
            report[f'Delta_mIoU_{platform}'] = delta_miou
            total_delta += (delta_f1 + delta_miou)
            
        rs_plat = 1.0 - (1.0 / (2.0 * num_platforms)) * total_delta
        report["RS_plat"] = rs_plat
        return report

    @staticmethod
    def compute_metadata_resilience(perf_with_meta, perf_without_meta):
        rs_meta_f1 = perf_without_meta["F1_Score"] / (perf_with_meta["F1_Score"] + 1e-7)
        rs_meta_miou = perf_without_meta["mIoU"] / (perf_with_meta["mIoU"] + 1e-7)
        
        return {
            "RS_meta_F1": rs_meta_f1,
            "RS_meta_mIoU": rs_meta_miou,
            "Target_Passed": rs_meta_f1 >= 0.9 and rs_meta_miou >= 0.9
        }

# =====================================================================
# VERIFICATION PIPELINE HARNESS (MOCK HARDWARE SIMULATION RUN)
# =====================================================================
if __name__ == "__main__":
    print("\n=== INITIALIZING FAKESHIELD-T FULL SYSTEM EXPERIMENTAL TEST ===")
    
    model = FakeShieldT(metadata_dim=16).to(device)
    model.eval()
    
    mock_images = torch.randn(4, 3, 224, 224).to(device)
    mock_metadata = torch.randn(4, 16).to(device)
    mock_meta_flags = torch.ones(4).to(device) 
    
    mock_labels = torch.tensor([1, 0, 1, 0]).to(device) 
    mock_gt_masks = torch.randint(0, 2, (4, 1, 224, 224)).float().to(device)
    
    # Execution Step A: Baseline Pristine Settings Evaluation
    with torch.no_grad():
        logits_clean, masks_clean = model(mock_images, mock_metadata, mock_meta_flags)
        
    clean_metrics = FakeShieldEvaluationEngine.evaluation_report(
        logits_clean, mock_labels, masks_clean, mock_gt_masks
    )
    print(f"\n[+] Core Task Performance (Clean Target):")
    print(f"    - Detection F1-Score: {clean_metrics['F1_Score']:.4f}")
    print(f"    - Localization mIoU:  {clean_metrics['mIoU']:.4f}")

    # Execution Step B: Social Networks Anti-Forensics Laundering Simulation
    platform_simulations = ["Telegram", "WhatsApp", "Facebook"]
    platform_performance = {}
    
    for platform in platform_simulations:
        perturbed_images = mock_images + torch.randn_like(mock_images) * 0.1
        with torch.no_grad():
            logits_p, masks_p = model(perturbed_images, mock_metadata, mock_meta_flags)
        platform_performance[platform] = FakeShieldEvaluationEngine.evaluation_report(
            logits_p, mock_labels, masks_p, mock_gt_masks
        )
        
    robustness_report = FakeShieldEvaluationEngine.compute_robustness_metrics(
        clean_metrics, platform_performance
    )
    print(f"\n[+] Platform Robustness Metric Gaps:")
    for k, v in robustness_report.items():
        print(f"    - {k}: {v:.4f}")

    # Execution Step C: Stripped Structural EXIF Channel Analysis via Missing Token
    mock_missing_flags = torch.zeros(4).to(device)
    with torch.no_grad():
        logits_stripped, masks_stripped = model(mock_images, mock_metadata, mock_missing_flags)
        
    stripped_metrics = FakeShieldEvaluationEngine.evaluation_report(
        logits_stripped, mock_labels, masks_stripped, mock_gt_masks
    )
    
    resilience_report = FakeShieldEvaluationEngine.compute_metadata_resilience(
        clean_metrics, stripped_metrics
    )
    print(f"\n[+] Metadata Resilience Assessment Metrics:")
    print(f"    - RS_meta (F1 Index):   {resilience_report['RS_meta_F1']:.4f}")
    print(f"    - RS_meta (mIoU Index): {resilience_report['RS_meta_mIoU']:.4f}")
    print(f"    - Meets Thesis Target (RS_meta >= 0.90): {resilience_report['Target_Passed']}")
    print("\n=== SYSTEM VERIFICATION COMPLETE WITHOUT ERRORS ===")

=== STARTING DATASET SCAN AND SUBSAMPLING ===
Found 1000000 files in DF2023 Train
Found 5000 files in DF2023 Val
Found 10000 files in SIDA category: full_synthetic
Found 10000 files in SIDA category: masks
Found 10000 files in SIDA category: real
Found 10000 files in SIDA category: tampered
Found 10 parquet shards in So-Fake: train
Found 4 parquet shards in So-Fake: validation
Found 4 parquet shards in So-Fake: ood_test

=== FINAL GENERATED METADATA MATRIX ===
source_dataset    split
DF2023_Train      train    10000
DF2023_Val        test      1500
                  val       1500
sida-forensics    test       900
                  train     4200
                  val        900
so-fake-combined  test      1912
                  train     2000
                  val       2000
dtype: int64

=== INITIALIZING FAKESHIELD-T FULL SYSTEM EXPERIMENTAL TEST ===

[+] Core Task Performance (Clean Target):
    - Detection F1-Score: 0.0000
    - Localization mIoU:  0.4975

[+] Platform Robustness Me